In [2]:
%run "///home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/simulation/impport_packages.ipynb"    #import all necessary packages - numpy, pandas etc
%run "///home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/simulation/simulation_class.ipynb"    #import all necessary packages - numpy, pandas etc
%run "///home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/simulation/create_world.ipynb"

#### Start -3

In [5]:
dim_objnames_epi = {}
folder_path = '/home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/results/long_time/start-3/epi'
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".pkl"):
        
        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}")
        
        # Load the pickle file
        with open(file_path, "rb") as file:
            data = pickle.load(file)
            
        nof_dims = re.search(r'_dims(\d+)', filename).group(1)

        
        name= f'epi_{nof_dims}'
        dim_objnames_epi[name] = data
        
        print(f"Loaded {filename} into {name}")
        
dim_objnames_onlygen = {}
folder_path = '/home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/results/long_time/start-3/onlygen'
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".pkl"):
        
        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}")
        
        # Load the pickle file
        with open(file_path, "rb") as file:
            data = pickle.load(file)
            
        nof_dims = re.search(r'_dims(\d+)', filename).group(1)

        
        name= f'onlygen_{nof_dims}'
        dim_objnames_onlygen[name] = data
        
        print(f"Loaded {filename} into {name}")

Loading: gens10000_start-3.0_dims1.pkl
Loaded gens10000_start-3.0_dims1.pkl into epi_1
Loading: gens10000_start-3.0_dims10.pkl
Loaded gens10000_start-3.0_dims10.pkl into epi_10
Loading: gens10000_start-3.0_dims5.pkl
Loaded gens10000_start-3.0_dims5.pkl into epi_5
Loading: gens10000_start-3.0_dims10_onlygen.pkl
Loaded gens10000_start-3.0_dims10_onlygen.pkl into onlygen_10
Loading: gens10000_start-3.0_dims1_onlygen.pkl
Loaded gens10000_start-3.0_dims1_onlygen.pkl into onlygen_1
Loading: gens10000_start-3.0_dims5_onlygen.pkl
Loaded gens10000_start-3.0_dims5_onlygen.pkl into onlygen_5


In [ ]:
sorted_epi_keys = sorted(dim_objnames_epi.keys(), key=lambda x: int(re.findall(r'\d+', x)[0]))
sorted_og_keys = sorted(dim_objnames_onlygen.keys(), key=lambda x: int(re.findall(r'\d+', x)[0]))
# Get all keys 
keys = dim_objnames_epi[sorted_epi_keys[0]]['results_array_1'][0].keys()
keys

In [ ]:
start = 1
concatenated_results_epi = {}
concatenated_results_gen = {}

for array_epi, array_gen in zip(sorted_epi_keys, sorted_og_keys):
    
    nof_dims = re.findall(r'\d+', array_epi)[0]
    
    
    array_name_epi = next(iter(dim_objnames_epi[array_epi].keys()))
    array_name_gen = next(iter(dim_objnames_onlygen[array_gen].keys()))

    
    
    for k in keys:
        concatenated_results_epi[k] = np.vstack([r[k] for r in dim_objnames_epi[array_epi][array_name_epi]])
        
    for k in keys:
        concatenated_results_gen[k] = np.vstack([r[k] for r in dim_objnames_onlygen[array_gen][array_name_gen]])
        
    # Extract variables
    data = concatenated_results_epi
    data_gen = concatenated_results_gen
    
    
    meanmemory_p = data['meanmemory_p'][:, :-1]
    meanneutral_p = data['meanneutral_p'][:, :-1]


    nof_scenarios = meanmemory_p.shape[0]
    maxgen = meanmemory_p.shape[1]

        # Extract rho for each scenario
    rho_values = data['rho_m_alpha_beta'][:, 0]   # one rho per scenario

    # Create figure
    fig, ax = plt.subplots(1, 2, figsize=(18, 5), sharex=True)

    # Normalize rho range → [0, 1]
    norm = mcolors.Normalize(vmin=-1, vmax=1)
    cmap = cm.get_cmap("spring")

    # Loop over scenarios
    for i in range(start, nof_scenarios):
        rho = rho_values[i]           # pick rho for this scenario
        color = cmap(norm(rho))       # convert rho → color

        ax[0].plot(meanmemory_p[i, :], color=color, alpha=0.8)
        ax[1].plot(meanneutral_p[i, :], color=color, alpha=0.8)

    # Axis formatting
    ax[0].set_title("Mean Memory ", fontsize=18)
    ax[0].set_ylabel("Mean memory", fontsize=18)
    ax[0].set_xlabel("Generation",  fontsize=18)
    ax[0].set_xlim(1, maxgen)
    ax[0].set_ylim(0, 1)

    ax[1].set_title("Mean Neutral", fontsize=18)
    ax[1].set_ylabel("Mean neutral",  fontsize=18)
    ax[1].set_xlabel("Generation", fontsize=18)
    ax[1].set_xlim(1, maxgen)
    ax[1].set_ylim(0, 1)
    ax[0].tick_params(axis='both', labelsize=14)
    ax[1].tick_params(axis='both', labelsize=14)
    
    #  Add a colorbar based on rho values
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # required
    cbar = fig.colorbar(sm, ax=ax[0], label="Autocorrelation of environement (rho)", ticks=[-1, -0.5, 0,  0.5,  1.0])
    cbar = fig.colorbar(sm, ax=ax[1], label="Autocorrelation of environement (rho)", ticks=[-1, -0.5, 0,  0.5,  1.0])
    fig.suptitle(f"Nof dimensions: {nof_dims}, Scenarios {start}–{nof_scenarios - 1}", fontsize=20)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    ##########
 
    rho_ext, m_ext = np.meshgrid(data['rho_m_alpha_beta'][:,0],data['rho_m_alpha_beta'][:,1])

    df = pd.DataFrame({
        
    'rho': data['rho_m_alpha_beta'][:,0],
    'm': data['rho_m_alpha_beta'][:,1],
    'fitness':np.mean(data['meanw'][:,-50:], axis=1),
    'epi_mem_p': np.mean(data['meanmemory_p'][:,-50:], axis=1),
    'neutral_trait_p' : np.mean(data['meanneutral_p'][:,-50:], axis=1 )
    })
    
    df_gen = pd.DataFrame({
        
    'rho': data_gen['rho_m_alpha_beta'][:,0],
    'm': data_gen['rho_m_alpha_beta'][:,1],
    'fitness':np.mean(data_gen['meanw'][:,-50:], axis=1)
    })
    


    # Extract the two traits
    epi = df['epi_mem_p']
    neu = df['neutral_trait_p']

    # Define discrete bins
    bins = np.linspace(0, 1, 21)   # 20 bins
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # Compute counts
    counts_epi, _ = np.histogram(epi, bins=bins)
    counts_neu, _ = np.histogram(neu, bins=bins)

    # Convert to frequencies
    freq_epi = counts_epi / counts_epi.sum()
    freq_neu = counts_neu / counts_neu.sum()

    # Width of each bar
    height = 0.02  # vertical thickness

    # Plot horizontal bars
    fig, ax = plt.subplots(figsize=(10, 6))

    # Side-by-side: shift up/down
    offset = height * 1.2

    # Epigenetic memory (upper)
    ax.barh(bin_centers + offset/2, freq_epi, height=height, color='tab:blue', alpha=0.8, label='Epigenetic memory strength')

    # Neutral trait (lower)
    ax.barh(bin_centers - offset/2, freq_neu, height=height, color='tab:orange', alpha=0.8, label='Neutral trait' )

    # Labels & formatting
    ax.set_xlabel("Proportion of total count", fontsize=20)
    ax.set_ylabel("Evolved epigenetic memory \n(mean of 50 generations)", fontsize=20)

    ax.set_title(
    f"Comparative horizontal histogram of final evolved trait (dimensions = {nof_dims})",
    fontsize=24, pad=15  
)

    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.legend(fontsize=20)

    plt.tight_layout()
    plt.show()



    fig, ax = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Fitness and Evolved Epigenetic memory strength, no. of dimentions {nof_dims}", fontsize=16)
    fig.subplots_adjust(top=0.90)   # more space for the suptitle
    cmap1 = plt.cm.copper.copy()  # copy to modify safely
    cmap1.set_bad(color='gray')    # gray for masked values
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)  # consistent color range
    
    cmap2 = plt.cm.viridis.copy()  # copy to modify safely
    cmap2.set_bad(color='gray')    # gray for masked values
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)  # consistent color range
    
    
    # --- (0,0) fitness ---
    pivoted_fitness = df.pivot_table(
        index='m',
        columns='rho',
        values='fitness'
    ).sort_index().sort_index(axis=1)

    pcm1 = ax[0,0].pcolormesh(
        pivoted_fitness.columns,
        pivoted_fitness.index,
        pivoted_fitness.values,
        cmap=cmap1,
        norm=norm,
        shading='auto'
    )
    cbar1 = fig.colorbar(pcm1, ax=ax[0,0], label='Fitness (W) with epigenetic memory', 
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar1.ax.tick_params(labelsize=14)

    ax[0,0].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[0,0].set_ylim(0.5,1)
    ax[0,0].set_xlabel('Autocorrelation of environement (rho) ', fontsize=12)
    ax[0,0].set_title('Fitness of trait w epi-memory \nvs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (0,1) genetic fitness ---
    pivoted_gen = df_gen.pivot_table(
        index='m',
        columns='rho',
        values='fitness'
    ).sort_index().sort_index(axis=1)

    pcm2 = ax[0,1].pcolormesh(
        pivoted_gen.columns,
        pivoted_gen.index,
        pivoted_gen.values,
        cmap=cmap1,
        norm=norm,
        shading='auto'
    )
    cbar2 = fig.colorbar(pcm2, ax=ax[0,1], 
                        label=' Fitness (W) with only genetics',
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar2.ax.tick_params(labelsize=14)

    ax[0,1].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[0,1].set_ylim(0.5,1)
    ax[0,1].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[0,1].set_title(' Fitness of genetic trait \nvs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (1,0) epimem ---
    pivoted_epi = df.pivot_table(
        index='m',
        columns='rho',
        values='epi_mem_p'
    ).sort_index().sort_index(axis=1)

    pcm3 = ax[1,0].pcolormesh(
        pivoted_epi.columns,
        pivoted_epi.index,
        pivoted_epi.values,
        cmap=cmap2,
        norm=norm,
        shading='auto'
    )
    cbar3 = fig.colorbar(pcm3, ax=ax[1,0], label=r'Evolved Epi-memory strength',
                        ticks=[0, 0.25, 0.5, 0.75, 1.0])
    cbar3.ax.tick_params(labelsize=14)

    ax[1,0].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[1,0].set_ylim(0.5,1)
    ax[1,0].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[1,0].set_title('Epi-memory vs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (1,1) neutral ---
    pivoted_neutral = df.pivot_table(
        index='m',
        columns='rho',
        values='neutral_trait_p'
    ).sort_index().sort_index(axis=1)

    pcm4 = ax[1,1].pcolormesh(
        pivoted_neutral.columns,
        pivoted_neutral.index,
        pivoted_neutral.values,
        cmap=cmap2,
        norm=norm,
        shading='auto'
    )
    cbar4 = fig.colorbar(pcm4, ax=ax[1,1], label='Neutral trait',
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar4.ax.tick_params(labelsize=14)

    ax[1,1].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[1,1].set_ylim(0.5,1)
    ax[1,1].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[1,1].set_title('Neutral trait vs Autocorrelation and mean time spent in A', fontsize=14,  pad=15   )
    
    

    plt.tight_layout()
    plt.show()
    
    
    fig, ax = plt.subplots(figsize=(8, 5))
    plt.scatter( df['neutral_trait_p'], df['epi_mem_p'] )
    plt.title(f"epimem vs neutral trait, no. of dimentions = {nof_dims}")
    plt.plot((0,1), (0,1), 'r--', alpha=0.8)  # red dashed line
    
    plt.ylabel("epigenetic memory phenotype")
    plt.xlabel("neutral phenotype")
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.show()


#### Starting point 0

In [ ]:
dim_objnames_epi = {}
folder_path = '/home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/results/long_time/start0/epi'
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".pkl"):
        
        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}")
        
        # Load the pickle file
        with open(file_path, "rb") as file:
            data = pickle.load(file)
            
        nof_dims = re.search(r'_dims(\d+)', filename).group(1)

        
        name= f'epi_{nof_dims}'
        dim_objnames_epi[name] = data
        
        print(f"Loaded {filename} into {name}")
        
dim_objnames_onlygen = {}
folder_path = '/home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/results/long_time/start0/onlygen'
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".pkl"):
        
        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}")
        
        # Load the pickle file
        with open(file_path, "rb") as file:
            data = pickle.load(file)
            
        nof_dims = re.search(r'_dims(\d+)', filename).group(1)

        
        name= f'onlygen_{nof_dims}'
        dim_objnames_onlygen[name] = data
        
        print(f"Loaded {filename} into {name}")

In [ ]:

sorted_epi_keys = sorted(dim_objnames_epi.keys(), key=lambda x: int(re.findall(r'\d+', x)[0]))
sorted_og_keys = sorted(dim_objnames_onlygen.keys(), key=lambda x: int(re.findall(r'\d+', x)[0]))
# Get all keys 
keys = dim_objnames_epi[sorted_epi_keys[0]]['results_array_1'][0].keys()
keys

In [ ]:
start = 1
concatenated_results_epi = {}
concatenated_results_gen = {}

for array_epi, array_gen in zip(sorted_epi_keys, sorted_og_keys):
    
    nof_dims = re.findall(r'\d+', array_epi)[0]
    
    
    array_name_epi = next(iter(dim_objnames_epi[array_epi].keys()))
    array_name_gen = next(iter(dim_objnames_onlygen[array_gen].keys()))

    
    for k in keys:
        concatenated_results_epi[k] = np.vstack([r[k] for r in dim_objnames_epi[array_epi][array_name_epi] if r[k][0]['rho_m_alpha_beta'][1] >= 0.5])
        
    for k in keys:
        concatenated_results_gen[k] = np.vstack([r[k] for r in dim_objnames_onlygen[array_gen][array_name_gen] if r[0]['rho_m_alpha_beta'][1] >= 0.5])
        
    # Extract variables
    data = concatenated_results_epi
    data_gen = concatenated_results_gen
    
    
    meanmemory_p = data['meanmemory_p'][:, :-1]
    meanneutral_p = data['meanneutral_p'][:, :-1]


    nof_scenarios = meanmemory_p.shape[0]
    maxgen = meanmemory_p.shape[1]

        # Extract rho for each scenario
    rho_values = data['rho_m_alpha_beta'][:, 0]   # one rho per scenario
    m_vals = data['rho_m_alpha_beta'][:, 1]
    
    # Create figure
    fig, ax = plt.subplots(2, 2, figsize=(25, 10), sharex=True, sharey=True)

    # Normalize rho range → [0, 1]
    norm_rho = mcolors.Normalize(vmin=-1, vmax=1)
    norm_m = mcolors.Normalize(vmin=0, vmax=1)
    cmap_rho = cm.get_cmap("spring")
    cmap_m = cm.get_cmap("cividis")
    
    # Loop over scenarios
    for i in range(start, nof_scenarios):
        rho = rho_values[i]           # pick rho for this scenario
        m = m_vals[i]
        color_rho = cmap_rho(norm_rho(rho))       # convert rho → color
        color_m = cmap_m(norm_m(m)) 
        
        ax[0,0].plot(meanmemory_p[i, :], color=color_rho, alpha=0.8)
        ax[0,1].plot(meanneutral_p[i, :], color=color_rho, alpha=0.8)
        ax[1,0].plot(meanmemory_p[i, :], color=color_m, alpha=0.8)
        ax[1,1].plot(meanneutral_p[i, :], color=color_m, alpha=0.8)

    # Axis formatting
    ax[0,0].set_title("Mean Memory ", fontsize=18)
    ax[0,0].set_ylabel("Mean memory", fontsize=18)
    ax[0,0].set_xlabel("Generation",  fontsize=18)
    ax[0,0].set_xlim(1, maxgen)
    ax[0,0].set_ylim(0, 1)

    ax[0,1].set_title("Mean Neutral", fontsize=18)
    ax[0,1].set_ylabel("Mean neutral",  fontsize=18)
    ax[0,1].set_xlabel("Generation", fontsize=18)
    ax[0,1].set_xlim(1, maxgen)
    ax[0,1].set_ylim(0, 1)
    
    ax[1,0].set_title("Mean Memory ", fontsize=18)
    ax[1,0].set_ylabel("Mean memory", fontsize=18)
    ax[1,0].set_xlabel("Generation",  fontsize=18)
    ax[1,0].set_xlim(1, maxgen)
    ax[1,0].set_ylim(0, 1)
    
    ax[1,1].set_title("Mean Neutral", fontsize=18)
    ax[1,1].set_ylabel("Mean neutral",  fontsize=18)
    ax[1,1].set_xlabel("Generation", fontsize=18)
    ax[1,1].set_xlim(1, maxgen)
    ax[1,1].set_ylim(0, 1)
    ax[1,1].tick_params(axis='both', labelsize=14)
    ax[1,1].tick_params(axis='both', labelsize=14)
    
    #  Add a colorbar based on rho values
    sm = cm.ScalarMappable(cmap=cmap_rho, norm=norm_rho)
    sm.set_array([])  # required
    
    cbar = fig.colorbar(sm, ax=ax[0,0], label="Autocorrelation of environement (rho)", ticks=[-1, -0.5, 0,  0.5,  1.0])
    cbar = fig.colorbar(sm, ax=ax[0,1], label="Autocorrelation of environement (rho)", ticks=[-1, -0.5, 0,  0.5,  1.0])
    
    sm = cm.ScalarMappable(cmap=cmap_m, norm=norm_m)
    sm.set_array([])  # required
    
    cbar = fig.colorbar(sm, ax=ax[1,0], label="mean time spent (m)", ticks=[ 0,  0.5,  1.0])
    cbar = fig.colorbar(sm, ax=ax[1,1], label="mean time spent (m)", ticks=[ 0,  0.5,  1.0])
    
    
    fig.suptitle(f"Nof dimensions: {nof_dims}, Scenarios {start}–{nof_scenarios - 1}", fontsize=20)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    ##########
 
    rho_ext, m_ext = np.meshgrid(data['rho_m_alpha_beta'][:,0],data['rho_m_alpha_beta'][:,1])

    df = pd.DataFrame({
        
    'rho': data['rho_m_alpha_beta'][:,0],
    'm': data['rho_m_alpha_beta'][:,1],
    'fitness':np.mean(data['meanw'][:,-50:], axis=1),
    'epi_mem_p': np.mean(data['meanmemory_p'][:,-50:], axis=1),
    'neutral_trait_p' : np.mean(data['meanneutral_p'][:,-50:], axis=1 )
    })
    
    df_gen = pd.DataFrame({
        
    'rho': data_gen['rho_m_alpha_beta'][:,0],
    'm': data_gen['rho_m_alpha_beta'][:,1],
    'fitness':np.mean(data_gen['meanw'][:,-50:], axis=1)
    })
    


    # Extract the two traits
    epi = df['epi_mem_p']
    neu = df['neutral_trait_p']

    # Define discrete bins
    bins = np.linspace(0, 1, 21)   # 20 bins
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # Compute counts
    counts_epi, _ = np.histogram(epi, bins=bins)
    counts_neu, _ = np.histogram(neu, bins=bins)

    # Convert to frequencies
    freq_epi = counts_epi / counts_epi.sum()
    freq_neu = counts_neu / counts_neu.sum()

    # Width of each bar
    height = 0.02  # vertical thickness

    # Plot horizontal bars
    fig, ax = plt.subplots(figsize=(10, 6))

    # Side-by-side: shift up/down
    offset = height * 1.2

    # Epigenetic memory (upper)
    ax.barh(bin_centers + offset/2, freq_epi, height=height, color='tab:blue', alpha=0.8, label='Epigenetic memory strength')

    # Neutral trait (lower)
    ax.barh(bin_centers - offset/2, freq_neu, height=height, color='tab:orange', alpha=0.8, label='Neutral trait' )

    # Labels & formatting
    ax.set_xlabel("Proportion of total count", fontsize=20)
    ax.set_ylabel("Evolved epigenetic memory \n(mean of 50 generations)", fontsize=20)

    ax.set_title(
    f"Comparative horizontal histogram of final evolved trait (dimensions = {nof_dims})",
    fontsize=24, pad=15  
)

    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.legend(fontsize=20)

    plt.tight_layout()
    plt.show()



    fig, ax = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Fitness and Evolved Epigenetic memory strength, no. of dimentions {nof_dims}", fontsize=16)
    fig.subplots_adjust(top=0.90)   # more space for the suptitle
    cmap1 = plt.cm.copper.copy()  # copy to modify safely
    cmap1.set_bad(color='gray')    # gray for masked values
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)  # consistent color range
    
    cmap2 = plt.cm.viridis.copy()  # copy to modify safely
    cmap2.set_bad(color='gray')    # gray for masked values
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)  # consistent color range
    
    
    # --- (0,0) fitness ---
    pivoted_fitness = df.pivot_table(
        index='m',
        columns='rho',
        values='fitness'
    ).sort_index().sort_index(axis=1)

    pcm1 = ax[0,0].pcolormesh(
        pivoted_fitness.columns,
        pivoted_fitness.index,
        pivoted_fitness.values,
        cmap=cmap1,
        norm=norm_rho,
        shading='auto'
    )
    cbar1 = fig.colorbar(pcm1, ax=ax[0,0], label='Fitness (W) with epigenetic memory', 
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar1.ax.tick_params(labelsize=14)

    ax[0,0].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[0,0].set_ylim(0,1)
    ax[0,0].set_xlabel('Autocorrelation of environement (rho) ', fontsize=12)
    ax[0,0].set_title('Fitness of trait w epi-memory \nvs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (0,1) genetic fitness ---
    pivoted_gen = df_gen.pivot_table(
        index='m',
        columns='rho',
        values='fitness'
    ).sort_index().sort_index(axis=1)

    pcm2 = ax[0,1].pcolormesh(
        pivoted_gen.columns,
        pivoted_gen.index,
        pivoted_gen.values,
        cmap=cmap1,
        norm=norm_rho,
        shading='auto'
    )
    cbar2 = fig.colorbar(pcm2, ax=ax[0,1], 
                        label=' Fitness (W) with only genetics',
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar2.ax.tick_params(labelsize=14)

    ax[0,1].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[0,1].set_ylim(0,1)
    ax[0,1].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[0,1].set_title(' Fitness of genetic trait \nvs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (1,0) epimem ---
    pivoted_epi = df.pivot_table(
        index='m',
        columns='rho',
        values='epi_mem_p'
    ).sort_index().sort_index(axis=1)

    pcm3 = ax[1,0].pcolormesh(
        pivoted_epi.columns,
        pivoted_epi.index,
        pivoted_epi.values,
        cmap=cmap2,
        norm=norm_m,
        shading='auto'
    )
    cbar3 = fig.colorbar(pcm3, ax=ax[1,0], label=r'Evolved Epi-memory strength',
                        ticks=[0, 0.25, 0.5, 0.75, 1.0])
    cbar3.ax.tick_params(labelsize=14)

    ax[1,0].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[1,0].set_ylim(0,1)
    ax[1,0].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[1,0].set_title('Epi-memory vs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (1,1) neutral ---
    pivoted_neutral = df.pivot_table(
        index='m',
        columns='rho',
        values='neutral_trait_p'
    ).sort_index().sort_index(axis=1)

    pcm4 = ax[1,1].pcolormesh(
        pivoted_neutral.columns,
        pivoted_neutral.index,
        pivoted_neutral.values,
        cmap=cmap2,
        norm=norm_m,
        shading='auto'
    )
    cbar4 = fig.colorbar(pcm4, ax=ax[1,1], label='Neutral trait',
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar4.ax.tick_params(labelsize=14)

    ax[1,1].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[1,1].set_ylim(0,1)
    ax[1,1].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[1,1].set_title('Neutral trait vs Autocorrelation and mean time spent in A', fontsize=14,  pad=15   )
    
    

    plt.tight_layout()
    plt.show()
    
    
    fig, ax = plt.subplots(figsize=(8, 5))
    plt.scatter( df['neutral_trait_p'], df['epi_mem_p'] )
    plt.title(f"epimem vs neutral trait, no. of dimentions = {nof_dims}")
    plt.plot((0,1), (0,1), 'r--', alpha=0.8)  # red dashed line
    
    plt.ylabel("epigenetic memory phenotype")
    plt.xlabel("neutral phenotype")
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.show()


### Starting point 3

In [ ]:
dim_objnames_epi = {}
folder_path = '/home/usriniva/uller_modified/discrete_time/Hanna_simu/results/long_time/start3/epi'
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".pkl"):
        
        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}")
        
        # Load the pickle file
        with open(file_path, "rb") as file:
            data = pickle.load(file)
            
        nof_dims = re.search(r'_dims(\d+)', filename).group(1)

        
        name= f'epi_{nof_dims}'
        dim_objnames_epi[name] = data
        
        print(f"Loaded {filename} into {name}")
        
dim_objnames_onlygen = {}
folder_path = '/home/usriniva/uller_modified/discrete_time/Hanna_simu/results/long_time/start3/onlygen'
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".pkl"):
        
        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}")
        
        # Load the pickle file
        with open(file_path, "rb") as file:
            data = pickle.load(file)
            
        nof_dims = re.search(r'_dims(\d+)', filename).group(1)

        
        name= f'onlygen_{nof_dims}'
        dim_objnames_onlygen[name] = data
        
        print(f"Loaded {filename} into {name}")

sorted_epi_keys = sorted(dim_objnames_epi.keys(), key=lambda x: int(re.findall(r'\d+', x)[0]))
sorted_og_keys = sorted(dim_objnames_onlygen.keys(), key=lambda x: int(re.findall(r'\d+', x)[0]))
# Get all keys 
keys = dim_objnames_epi[sorted_epi_keys[0]]['results_array_1'][0].keys()
keys

In [ ]:
start = 1
concatenated_results_epi = {}
concatenated_results_gen = {}

for array_epi, array_gen in zip(sorted_epi_keys, sorted_og_keys):
    
    nof_dims = re.findall(r'\d+', array_epi)[0]
    
    
    array_name_epi = next(iter(dim_objnames_epi[array_epi].keys()))
    array_name_gen = next(iter(dim_objnames_onlygen[array_gen].keys()))

    
    
    for k in keys:
        concatenated_results_epi[k] = np.vstack([r[k] for r in dim_objnames_epi[array_epi][array_name_epi]])
        
    for k in keys:
        concatenated_results_gen[k] = np.vstack([r[k] for r in dim_objnames_onlygen[array_gen][array_name_gen]])
        
    # Extract variables
    data = concatenated_results_epi
    data_gen = concatenated_results_gen
    
    
    meanmemory_p = data['meanmemory_p'][:, :-1]
    meanneutral_p = data['meanneutral_p'][:, :-1]


    nof_scenarios = meanmemory_p.shape[0]
    maxgen = meanmemory_p.shape[1]

        # Extract rho for each scenario
    rho_values = data['rho_m_alpha_beta'][:, 0]   # one rho per scenario

    # Create figure
    fig, ax = plt.subplots(1, 2, figsize=(18, 5), sharex=True)

    # Normalize rho range → [0, 1]
    norm = mcolors.Normalize(vmin=-1, vmax=1)
    cmap = cm.get_cmap("spring")

    # Loop over scenarios
    for i in range(start, nof_scenarios):
        rho = rho_values[i]           # pick rho for this scenario
        color = cmap(norm(rho))       # convert rho → color

        ax[0].plot(meanmemory_p[i, :], color=color, alpha=0.8)
        ax[1].plot(meanneutral_p[i, :], color=color, alpha=0.8)

    # Axis formatting
    ax[0].set_title("Mean Memory ", fontsize=18)
    ax[0].set_ylabel("Mean memory", fontsize=18)
    ax[0].set_xlabel("Generation",  fontsize=18)
    ax[0].set_xlim(1, maxgen)
    ax[0].set_ylim(0, 1)

    ax[1].set_title("Mean Neutral", fontsize=18)
    ax[1].set_ylabel("Mean neutral",  fontsize=18)
    ax[1].set_xlabel("Generation", fontsize=18)
    ax[1].set_xlim(1, maxgen)
    ax[1].set_ylim(0, 1)
    ax[0].tick_params(axis='both', labelsize=14)
    ax[1].tick_params(axis='both', labelsize=14)
    
    #  Add a colorbar based on rho values
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # required
    cbar = fig.colorbar(sm, ax=ax[0], label="Autocorrelation of environement (rho)", ticks=[-1, -0.5, 0,  0.5,  1.0])
    cbar = fig.colorbar(sm, ax=ax[1], label="Autocorrelation of environement (rho)", ticks=[-1, -0.5, 0,  0.5,  1.0])
    fig.suptitle(f"Nof dimensions: {nof_dims}, Scenarios {start}–{nof_scenarios - 1}", fontsize=20)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    ##########
 
    rho_ext, m_ext = np.meshgrid(data['rho_m_alpha_beta'][:,0],data['rho_m_alpha_beta'][:,1])

    df = pd.DataFrame({
        
    'rho': data['rho_m_alpha_beta'][:,0],
    'm': data['rho_m_alpha_beta'][:,1],
    'fitness':np.mean(data['meanw'][:,-50:], axis=1),
    'epi_mem_p': np.mean(data['meanmemory_p'][:,-50:], axis=1),
    'neutral_trait_p' : np.mean(data['meanneutral_p'][:,-50:], axis=1 )
    })
    
    df_gen = pd.DataFrame({
        
    'rho': data_gen['rho_m_alpha_beta'][:,0],
    'm': data_gen['rho_m_alpha_beta'][:,1],
    'fitness':np.mean(data_gen['meanw'][:,-50:], axis=1)
    })
    


    # Extract the two traits
    epi = df['epi_mem_p']
    neu = df['neutral_trait_p']

    # Define discrete bins
    bins = np.linspace(0, 1, 21)   # 20 bins
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # Compute counts
    counts_epi, _ = np.histogram(epi, bins=bins)
    counts_neu, _ = np.histogram(neu, bins=bins)

    # Convert to frequencies
    freq_epi = counts_epi / counts_epi.sum()
    freq_neu = counts_neu / counts_neu.sum()

    # Width of each bar
    height = 0.02  # vertical thickness

    # Plot horizontal bars
    fig, ax = plt.subplots(figsize=(10, 6))

    # Side-by-side: shift up/down
    offset = height * 1.2

    # Epigenetic memory (upper)
    ax.barh(bin_centers + offset/2, freq_epi, height=height, color='tab:blue', alpha=0.8, label='Epigenetic memory strength')

    # Neutral trait (lower)
    ax.barh(bin_centers - offset/2, freq_neu, height=height, color='tab:orange', alpha=0.8, label='Neutral trait' )

    # Labels & formatting
    ax.set_xlabel("Proportion of total count", fontsize=20)
    ax.set_ylabel("Evolved epigenetic memory \n(mean of 50 generations)", fontsize=20)

    ax.set_title(
    f"Comparative horizontal histogram of final evolved trait (dimensions = {nof_dims})",
    fontsize=24, pad=15  
)

    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.legend(fontsize=20)

    plt.tight_layout()
    plt.show()



    fig, ax = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Fitness and Evolved Epigenetic memory strength, no. of dimentions {nof_dims}", fontsize=16)
    fig.subplots_adjust(top=0.90)   # more space for the suptitle
    cmap1 = plt.cm.copper.copy()  # copy to modify safely
    cmap1.set_bad(color='gray')    # gray for masked values
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)  # consistent color range
    
    cmap2 = plt.cm.viridis.copy()  # copy to modify safely
    cmap2.set_bad(color='gray')    # gray for masked values
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)  # consistent color range
    
    
    # --- (0,0) fitness ---
    pivoted_fitness = df.pivot_table(
        index='m',
        columns='rho',
        values='fitness'
    ).sort_index().sort_index(axis=1)

    pcm1 = ax[0,0].pcolormesh(
        pivoted_fitness.columns,
        pivoted_fitness.index,
        pivoted_fitness.values,
        cmap=cmap1,
        norm=norm,
        shading='auto'
    )
    cbar1 = fig.colorbar(pcm1, ax=ax[0,0], label='Fitness (W) with epigenetic memory', 
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar1.ax.tick_params(labelsize=14)

    ax[0,0].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[0,0].set_ylim(0.5,1)
    ax[0,0].set_xlabel('Autocorrelation of environement (rho) ', fontsize=12)
    ax[0,0].set_title('Fitness of trait w epi-memory \nvs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (0,1) genetic fitness ---
    pivoted_gen = df_gen.pivot_table(
        index='m',
        columns='rho',
        values='fitness'
    ).sort_index().sort_index(axis=1)

    pcm2 = ax[0,1].pcolormesh(
        pivoted_gen.columns,
        pivoted_gen.index,
        pivoted_gen.values,
        cmap=cmap1,
        norm=norm,
        shading='auto'
    )
    cbar2 = fig.colorbar(pcm2, ax=ax[0,1], 
                        label=' Fitness (W) with only genetics',
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar2.ax.tick_params(labelsize=14)

    ax[0,1].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[0,1].set_ylim(0.5,1)
    ax[0,1].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[0,1].set_title(' Fitness of genetic trait \nvs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (1,0) epimem ---
    pivoted_epi = df.pivot_table(
        index='m',
        columns='rho',
        values='epi_mem_p'
    ).sort_index().sort_index(axis=1)

    pcm3 = ax[1,0].pcolormesh(
        pivoted_epi.columns,
        pivoted_epi.index,
        pivoted_epi.values,
        cmap=cmap2,
        norm=norm,
        shading='auto'
    )
    cbar3 = fig.colorbar(pcm3, ax=ax[1,0], label=r'Evolved Epi-memory strength',
                        ticks=[0, 0.25, 0.5, 0.75, 1.0])
    cbar3.ax.tick_params(labelsize=14)

    ax[1,0].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[1,0].set_ylim(0.5,1)
    ax[1,0].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[1,0].set_title('Epi-memory vs Autocorrelation and mean time spent in A ', fontsize=14,   pad=15   )

    # --- (1,1) neutral ---
    pivoted_neutral = df.pivot_table(
        index='m',
        columns='rho',
        values='neutral_trait_p'
    ).sort_index().sort_index(axis=1)

    pcm4 = ax[1,1].pcolormesh(
        pivoted_neutral.columns,
        pivoted_neutral.index,
        pivoted_neutral.values,
        cmap=cmap2,
        norm=norm,
        shading='auto'
    )
    cbar4 = fig.colorbar(pcm4, ax=ax[1,1], label='Neutral trait',
                        ticks=[0.0, 0.25, 0.5, 0.75, 1.0])
    cbar4.ax.tick_params(labelsize=14)

    ax[1,1].set_ylabel('mean time spent in environment A (m)', fontsize=12)
    ax[1,1].set_ylim(0.5,1)
    ax[1,1].set_xlabel('Autocorrelation of environement (rho)', fontsize=12)
    ax[1,1].set_title('Neutral trait vs Autocorrelation and mean time spent in A', fontsize=14,  pad=15   )
    
    

    plt.tight_layout()
    plt.show()
    
    
    fig, ax = plt.subplots(figsize=(8, 5))
    plt.scatter( df['neutral_trait_p'], df['epi_mem_p'] )
    plt.title(f"epimem vs neutral trait, no. of dimentions = {nof_dims}")
    plt.plot((0,1), (0,1), 'r--', alpha=0.8)  # red dashed line
    
    plt.ylabel("epigenetic memory phenotype")
    plt.xlabel("neutral phenotype")
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.show()
